# DAY 8: Direct Preference Optimization (DPO)

## 🎯 Learning Objectives
In Phase 1, we used **Supervised Fine-Tuning (SFT)** to teach the model how to follow our target format.
In Phase 2, we move into **Preference Alignment**.

By the end of this session, you will be able to explain and execute:
1. **The Limitation of SFT**: SFT maximizes the log-likelihood $\log P(y_{\text{target}} | x)$ of positive examples, but cannot penalize undesirable behaviors or subtle hallucinations.
2. **The DPO Breakthrough**: Rafailov et al. (2023) showed that RLHF's complex reward-model-and-PPO loop can be mathematically re-parameterized into a single closed-form loss function.
3. **Pairwise Data Structure**: Formatting datasets into triplets: `(prompt, chosen, rejected)`.
4. **The $\beta$ Hyperparameter**: Controlling how much the aligned policy $\pi_\theta$ can drift from the reference policy $\pi_{\text{ref}}$.
5. **Reward Margins**: Calculating implicit rewards $r(x, y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$ and verifying $r(chosen) > r(rejected)$.

In [ ]:
# Install dependencies for Phase 2
!pip install -q trl peft transformers datasets accelerate bitsandbytes

In [ ]:
import os
import json
import random
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import DPOConfig, DPOTrainer

print("=" * 60)
print("🚀 DAY 8: DIRECT PREFERENCE OPTIMIZATION (DPO)")
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
print("=" * 60)

### Step 1: Generate Pairwise Preference Dataset (250 Samples across 5 Failure Modes)
DPO requires samples with two completions for each prompt:
- `chosen`: The ideal, concise, strictly formatted JSON target.
- `rejected`: Real-world base model failure modes (preambles, markdown backticks, key hallucinations, bad types, refusals).

In [ ]:
random.seed(42)

FIRST_NAMES = [
    "John", "Sarah", "Michael", "Emma", "David", "Emily", "James", "Olivia",
    "Robert", "Sophia", "William", "Ava", "Alexander", "Mia", "Daniel", "Charlotte",
    "Matthew", "Amelia", "Lucas", "Harper", "Benjamin", "Evelyn", "Elijah", "Abigail",
    "Liam", "Elizabeth", "Henry", "Sofia", "Oliver", "Victoria", "Noah", "Aria"
]

PRODUCTS = [
    ("laptops", 800, 2500), ("4K monitors", 300, 1200), ("mechanical keyboards", 50, 250),
    ("wireless mice", 25, 120), ("standing desks", 350, 1200), ("ergonomic office chairs", 150, 800),
    ("noise-canceling headphones", 120, 450), ("external SSDs (2TB)", 100, 300),
    ("laser printers", 180, 750), ("smartphones", 400, 1400), ("USB-C docking stations", 60, 250)
]

DELIVERY_DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday", "tomorrow", "this weekend"]

TEMPLATES = [
    "{customer} ordered {quantity} {product} for ${amount} and wants delivery on {delivery_day}.",
    "Please send {quantity} {product} to {customer}. The total is ${amount} and it needs to arrive on {delivery_day}.",
    "Invoice #{inv}: Customer {customer} purchased {quantity} units of {product}. Billed: ${amount}. Scheduled delivery: {delivery_day}.",
    "New transaction: {customer} - {product} (Qty: {quantity}) - Total: ${amount} - Requested: {delivery_day}.",
    "Dispatch request for {customer}: {quantity} {product} costing ${amount}, ensure arrival by {delivery_day}."
]

def generate_dpo_pair(idx):
    cust = random.choice(FIRST_NAMES)
    prod_name, min_p, max_p = random.choice(PRODUCTS)
    qty = random.randint(1, 10)
    amt = float(qty * random.randint(min_p // 10, max_p // 10) * 10)
    day = random.choice(DELIVERY_DAYS)
    inv = random.randint(10000, 99999)
    
    tmpl = random.choice(TEMPLATES)
    user_text = tmpl.format(customer=cust, product=prod_name, quantity=qty, amount=int(amt), delivery_day=day, inv=inv)
    
    # Target chosen: clean, exact JSON
    chosen_dict = {
        "customer": cust,
        "quantity": qty,
        "product": prod_name,
        "amount": amt,
        "delivery_day": day
    }
    chosen_str = json.dumps(chosen_dict)
    
    # 5 realistic failure modes for rejected:
    mode = idx % 5
    if mode == 0:
        # Chatty conversational preamble & postamble
        rejected_str = f"Sure! Here is the extracted order JSON you requested:\n```json\n{chosen_str}\n```\nLet me know if you need anything else!"
    elif mode == 1:
        # Wrapped in markdown code fences without preamble
        rejected_str = f"```json\n{chosen_str}\n```"
    elif mode == 2:
        # Key name hallucinations (e.g. name, qty, item, price, day)
        hal_dict = {"name": cust, "qty": qty, "item": prod_name, "price": amt, "day": day}
        rejected_str = json.dumps(hal_dict)
    elif mode == 3:
        # Type error: currency symbol in amount or string quantity
        bad_type_dict = {"customer": cust, "quantity": str(qty), "product": prod_name, "amount": f"${int(amt)}", "delivery_day": day}
        rejected_str = json.dumps(bad_type_dict)
    else:
        # Sycophantic conversational evasion
        rejected_str = f"I am processing your request. {cust} placed an order for {qty} {prod_name} totaling ${amt} for {day}."

    return {
        "prompt": f"Extract order into JSON: {user_text}",
        "chosen": chosen_str,
        "rejected": rejected_str
    }

# Generate 250 diverse pairwise preference samples
preference_data = [generate_dpo_pair(i) for i in range(250)]
dpo_dataset = Dataset.from_list(preference_data)

print(f"✅ Generated {len(dpo_dataset)} pairwise DPO comparison samples across 5 failure modes.")
print("\nSample Pair (Index 0 - Chatty Preamble Penalty):")
print("• Prompt   :", dpo_dataset[0]['prompt'])
print("• Chosen   :", dpo_dataset[0]['chosen'])
print("• Rejected :", dpo_dataset[0]['rejected'][:80], "...")

### Step 2: Configure Model & 4-Bit QLoRA for DPO (Model A: From Base)

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
compute_dtype = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto"
)
model.config.torch_dtype = torch.float16

# Prepares quantized layers and layer norms for stable gradient backprop
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
print("✅ Model A loaded (Base Model + DPO LoRA).")

### Step 3: Configure DPO Trainer

In [ ]:
dpo_config = DPOConfig(
    output_dir="models/dpo_checkpoints",
    beta=0.1,
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    logging_steps=10,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    save_strategy="no",
    report_to="none"
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_dataset,
    processing_class=tokenizer
)

print("✅ DPOTrainer successfully initialized on 250 pairwise samples.")

### Step 4: Execute DPO Training & Inspect Convergence

In [ ]:
print("Starting DPO Alignment Run (250 pairs x 3 epochs = 375 optimization steps)...")
train_result = dpo_trainer.train()
print("✅ DPO Training Finished!")

### Step 5: Evaluate Model A (DPO directly on Base)

In [ ]:
import numpy as np

DPO_ADAPTER_PATH = "models/adapters/qwen-1.5b-dpo-from-base"
dpo_trainer.save_model(DPO_ADAPTER_PATH)
tokenizer.save_pretrained(DPO_ADAPTER_PATH)
print(f"✅ Saved DPO Adapter to {DPO_ADAPTER_PATH}")

test_pairs = [generate_dpo_pair(i + 1000) for i in range(50)]
model.eval()
margins_base = []
chosen_wins_base = 0
beta = 0.1

for pair in test_pairs:
    p = pair["prompt"]
    c_text = f"{p}\n{pair['chosen']}"
    r_text = f"{p}\n{pair['rejected']}"
    
    inp_c = tokenizer(c_text, return_tensors="pt").to(model.device)
    inp_r = tokenizer(r_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        out_c = model(**inp_c)
        out_r = model(**inp_r)
        logp_c = torch.log_softmax(out_c.logits, dim=-1).gather(2, inp_c.input_ids.unsqueeze(-1)).squeeze(-1).sum().item()
        logp_r = torch.log_softmax(out_r.logits, dim=-1).gather(2, inp_r.input_ids.unsqueeze(-1)).squeeze(-1).sum().item()
        
        with model.disable_adapter():
            ref_out_c = model(**inp_c)
            ref_out_r = model(**inp_r)
            ref_logp_c = torch.log_softmax(ref_out_c.logits, dim=-1).gather(2, inp_c.input_ids.unsqueeze(-1)).squeeze(-1).sum().item()
            ref_logp_r = torch.log_softmax(ref_out_r.logits, dim=-1).gather(2, inp_r.input_ids.unsqueeze(-1)).squeeze(-1).sum().item()
    
    r_c = beta * (logp_c - ref_logp_c)
    r_r = beta * (logp_r - ref_logp_r)
    m = r_c - r_r
    margins_base.append(m)
    if r_c > r_r:
        chosen_wins_base += 1

print("=" * 65)
print("📊 MODEL A: DPO DIRECTLY FROM BASE MODEL")
print("=" * 65)
print(f"Preference Accuracy : {chosen_wins_base / len(test_pairs) * 100:.2f}%")
print(f"Mean Reward Margin  : {np.mean(margins_base):+.4f}")
print("=" * 65)

### Step 6: Train Model B (DPO on top of SFT Model)
Now we execute the proper **2-Stage Industrial Pipeline**: Pretrained $\to$ SFT $\to$ DPO.
If the merged SFT model does not exist on disk, we auto-merge the SFT adapter from `models/adapters/qwen-1.5b-order-extractor`.

In [ ]:
MERGED_PATH = "models/merged/qwen-1.5b-order-extractor"
ADAPTER_PATH = "models/adapters/qwen-1.5b-order-extractor"

# Ensure merged model directory exists on local disk
if not os.path.exists(MERGED_PATH) or not os.path.exists(os.path.join(MERGED_PATH, "config.json")):
    print("⏳ Merged model not found on Colab disk. Merging Day 4 SFT adapter now (~25 seconds)...")
    os.makedirs(MERGED_PATH, exist_ok=True)
    base_cpu = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct",
        torch_dtype=torch.float16,
        device_map="cpu"
    )
    sft_cpu = PeftModel.from_pretrained(base_cpu, ADAPTER_PATH)
    merged_cpu = sft_cpu.merge_and_unload()
    merged_cpu.save_pretrained(MERGED_PATH)
    tok_cpu = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
    tok_cpu.save_pretrained(MERGED_PATH)
    del base_cpu, sft_cpu, merged_cpu
    import gc; gc.collect()
    print(f"✅ Successfully created and saved merged SFT model to {MERGED_PATH}")
else:
    print(f"✅ Found existing merged SFT model at {MERGED_PATH}")

# Load SFT model for DPO
sft_tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH)
if sft_tokenizer.pad_token is None:
    sft_tokenizer.pad_token = sft_tokenizer.eos_token

sft_model = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    quantization_config=bnb_config,
    torch_dtype=compute_dtype,
    device_map="auto"
)
sft_model.config.torch_dtype = torch.float16
sft_model = prepare_model_for_kbit_training(sft_model)

sft_peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
sft_model = get_peft_model(sft_model, sft_peft_config)

dpo_config_sft = DPOConfig(
    output_dir="models/dpo_sft_checkpoints",
    beta=0.1,
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    logging_steps=10,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    save_strategy="no",
    report_to="none"
)

dpo_trainer_sft = DPOTrainer(
    model=sft_model,
    ref_model=None,
    args=dpo_config_sft,
    train_dataset=dpo_dataset,
    processing_class=sft_tokenizer
)

print("\n🚀 Training Model B: DPO on top of SFT Model...")
dpo_trainer_sft.train()
print("✅ DPO on SFT model training complete!")

### Step 7: Head-to-Head Comparison: Model A vs Model B

In [ ]:
DPO_SFT_ADAPTER_PATH = "models/adapters/qwen-1.5b-dpo-from-sft"
dpo_trainer_sft.save_model(DPO_SFT_ADAPTER_PATH)
sft_tokenizer.save_pretrained(DPO_SFT_ADAPTER_PATH)

sft_model.eval()
margins_sft = []
chosen_wins_sft = 0

for pair in test_pairs:
    p = pair["prompt"]
    c_text = f"{p}\n{pair['chosen']}"
    r_text = f"{p}\n{pair['rejected']}"
    
    inp_c = sft_tokenizer(c_text, return_tensors="pt").to(sft_model.device)
    inp_r = sft_tokenizer(r_text, return_tensors="pt").to(sft_model.device)
    
    with torch.no_grad():
        out_c = sft_model(**inp_c)
        out_r = sft_model(**inp_r)
        logp_c = torch.log_softmax(out_c.logits, dim=-1).gather(2, inp_c.input_ids.unsqueeze(-1)).squeeze(-1).sum().item()
        logp_r = torch.log_softmax(out_r.logits, dim=-1).gather(2, inp_r.input_ids.unsqueeze(-1)).squeeze(-1).sum().item()
        
        with sft_model.disable_adapter():
            ref_out_c = sft_model(**inp_c)
            ref_out_r = sft_model(**inp_r)
            ref_logp_c = torch.log_softmax(ref_out_c.logits, dim=-1).gather(2, inp_c.input_ids.unsqueeze(-1)).squeeze(-1).sum().item()
            ref_logp_r = torch.log_softmax(ref_out_r.logits, dim=-1).gather(2, inp_r.input_ids.unsqueeze(-1)).squeeze(-1).sum().item()
    
    r_c = beta * (logp_c - ref_logp_c)
    r_r = beta * (logp_r - ref_logp_r)
    m = r_c - r_r
    margins_sft.append(m)
    if r_c > r_r:
        chosen_wins_sft += 1

print("=" * 75)
print("🏆 HEAD-TO-HEAD COMPARISON: DPO FROM BASE vs DPO FROM SFT")
print("=" * 75)
print(f"{'Metric':<30} | {'Model A (Base + DPO)':<20} | {'Model B (SFT + DPO)':<20}")
print("-" * 75)
print(f"{'Preference Accuracy':<30} | {chosen_wins_base/len(test_pairs)*100:>18.2f}% | {chosen_wins_sft/len(test_pairs)*100:>18.2f}%")
print(f"{'Mean Reward Margin':<30} | {np.mean(margins_base):>19.4f} | {np.mean(margins_sft):>19.4f}")
print("=" * 75)

# Generation Test on Model B
inp_b = sft_tokenizer(test_input, return_tensors="pt").to(sft_model.device)
with torch.no_grad():
    gen_b = sft_model.generate(**inp_b, max_new_tokens=60, do_sample=False)
out_b = sft_tokenizer.decode(gen_b[0][inp_b.input_ids.shape[1]:], skip_special_tokens=True).strip()
print("\n--- MODEL B INFERENCE TEST ---")
print("Input :", test_input)
print("Output:", out_b)

---
## 🎓 Day 8 Interview Mastery: DPO

### 💼 3 Critical DPO Interview Questions:
1. **Why does DPO eliminate the need for a separate Reward Model?**
   - *Answer*: DPO uses the Bradley-Terry preference objective to prove that the optimal reward function $r^*(x, y)$ can be analytically expressed as $r^*(x,y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)$. Substituting this directly into the loss function eliminates the reward model entirely.
2. **What happens if $\beta$ is set too high or too low?**
   - *Answer*: If $\beta \to \infty$, the policy is heavily constrained and will not deviate from the reference model (under-fitting preference). If $\beta \to 0$, the model over-optimizes preference and collapses into repetitive or degenerate text.
3. **How does PEFT save VRAM in DPO?**
   - *Answer*: When using LoRA with `ref_model=None`, `DPOTrainer` does not allocate a separate reference model in VRAM. It simply disables the LoRA adapter to compute $\pi_{\text{ref}}$ forward passes on the fly!